# Insurance Cross Sell Targeting

Propensity modeling with top-k campaign targeting and business segmentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_insurance_data, basic_clean, stratified_split
from src.features import build_preprocessor, prepare_model_inputs
from src.modeling import (
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
)
from src.targeting import build_lift_gain_table, generate_batch_target_list
from src.evaluation import build_leaderboard

SEED = 42
PROJECT_NAME = 'insurance-cross-sell-targeting'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'insurance_cross_sell'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Prioritize outreach budget toward customers with strongest cross-sell propensity.

Primary metric: PR-AUC
Secondary: ROC-AUC
Tertiary: Precision@K / recall policy behavior

## 2) Dataset Access and Data Dictionary

Dataset: Playground Series S4E7 insurance cross-sell.

In [ ]:
df = load_insurance_data(RAW_DIR, sample_frac=0.08, random_state=SEED)
df.shape

## 3) Data Cleaning and Leakage Checks

- Remove duplicates
- Validate binary target
- Hold out realistic validation split

In [ ]:
df = basic_clean(df, target_col='Response')
train_df, holdout_df = stratified_split(df, target_col='Response', random_state=SEED)
train_df['Response'].mean(), holdout_df['Response'].mean()

## 4) Feature Engineering

- Numeric and categorical preprocessing
- Imputation and one-hot encoding for robust scoring

In [ ]:
preprocessor = build_preprocessor(train_df.drop(columns=['Response']))
X_train, X_holdout, y_train, y_holdout = prepare_model_inputs(
    train_df=train_df,
    holdout_df=holdout_df,
    target_col='Response',
    preprocessor=preprocessor,
)
X_train.shape, X_holdout.shape

## 5) Validation Strategy

- Stratified holdout
- Business-threshold and top-k evaluation
- Lift/gain analysis for campaign planning

In [ ]:
pd.DataFrame({
    'split': ['train', 'holdout'],
    'rows': [len(train_df), len(holdout_df)],
    'response_rate': [train_df['Response'].mean(), holdout_df['Response'].mean()]
})

## 6) Track 1: LazyPredict Discovery Lab

In [ ]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

## 7) Selection of Top 3 Eligible Models

In [ ]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table=lazy_table,
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    y_holdout=y_holdout,
    random_state=SEED,
)
eligible_table, top3_families

## 8) Track 2: Manual Engineering Lab

Includes ranking/top-k policy and threshold tuning.

In [ ]:
manual_results, manual_models, manual_scores = run_manual_engineering_track(
    top3_families=top3_families,
    X_train=X_train, y_train=y_train,
    X_holdout=X_holdout, y_holdout=y_holdout,
    random_state=SEED,
)
manual_results[['model_name', 'pr_auc', 'roc_auc', 'precision_at_20pct', 'threshold']]

## 9) Track 3: FLAML Optimization Lab

In [ ]:
flaml_result = run_flaml_track(
    X_train=X_train, y_train=y_train,
    X_holdout=X_holdout, y_holdout=y_holdout,
    time_budget=120, random_state=SEED
)
flaml_result

## 10) Track 4: PyCaret Experiment Lab

In [ ]:
pycaret_result = run_pycaret_track(
    train_df=train_df,
    holdout_df=holdout_df,
    target_col='Response',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_insurance_cross_sell',
)
pycaret_result

## 11) Unified Leaderboard and Final Model Ranking

In [ ]:
leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
)
leaderboard.head(10)

In [ ]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_insurance_cross_sell.csv', index=False)
leaderboard[['model_name', 'library_source', 'holdout_primary_metric', 'holdout_tertiary_metric', 'final_rank']].head(10)

## 12) Business Recommendation

State winner rationale, safer runner-up scenario, and excluded tradeoff.

## 13) Inference / Deployment Path

Generate batch target list by campaign budget.

In [ ]:
winner = leaderboard.sort_values('final_rank').iloc[0]
if winner['library_source'] == 'manual' and winner['model_name'] in manual_scores:
    scores = manual_scores[winner['model_name']]
elif winner['library_source'] == 'flaml':
    scores = flaml_result.get('scores', np.array([]))
else:
    scores = pycaret_result.get('scores', np.array([]))

if len(scores) != len(holdout_df):
    scores = np.resize(scores, len(holdout_df))

id_col = 'id' if 'id' in holdout_df.columns else holdout_df.columns[0]
target_list = generate_batch_target_list(holdout_df[[id_col]].copy(), scores, id_col=id_col, top_target_pct=0.2, hold_pct=0.3)
target_list.to_csv(ARTIFACT_DIR / 'target_list.csv', index=False)
target_list.head()

In [ ]:
lift_gain = build_lift_gain_table(holdout_df['Response'].astype(int).values, scores, n_bins=10)
lift_gain.to_csv(ARTIFACT_DIR / 'lift_gain_table.csv', index=False)
lift_gain

## 14) Monitoring / Drift / Retraining Plan

Track PR-AUC, precision@20%, capture rate in top deciles, and campaign uplift stability.

## 15) Limitations and Next Steps

- Add treatment uplift modeling
- Add contact policy constraints (frequency caps)
- Add fairness and segment-level calibration diagnostics